In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load the Dataset
# Replace 'customer_data.csv' with your actual file path
# Assumes the target variable is named 'Churn' (e.g., Yes/No or 1/0)
try:
    df = pd.read_csv("Data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: File not found. Please check the file path.")
    # Exiting or creating dummy data could go here; we'll assume df is loaded for the rest of the script.

# 2. Preprocessing
# Drop missing values (Alternatively, you could impute them using SimpleImputer)
df = df.dropna()

# Separate features (X) and target (y)
# Adjust 'Churn' if your target column has a different name
X = df.drop('Churn', axis=1)
y = df['Churn']

# Convert categorical target to numerical if it's not already (e.g., 'Yes'/'No' to 1/0)
y = y.map({'Yes': 1, 'No': 0, 1: 1, 0: 0}) 

# Identify categorical and numerical columns in features
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

# One-hot encode categorical variables
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# 3. Train-Test Split
# Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Feature Scaling
# Scaling numerical features is good practice, though less critical for Random Forests than for models like SVM or Logistic Regression
scaler = StandardScaler()

# Fit on training data, transform both training and testing data
# We only scale numerical columns to avoid messing up the binary one-hot encoded columns
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# 5. Initialize and Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
print("Training the model...")
model.fit(X_train, y_train)

# 6. Make Predictions
y_pred = model.predict(X_test)

# 7. Evaluate the Model
print("\n--- Model Evaluation ---")
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Optional: Feature Importance
# See which features were most influential in predicting churn
feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
print("\nTop 5 Most Important Features:")
print(feature_importances.nlargest(5))

Dataset loaded successfully.


C:\Users\Neelesh_New\AppData\Local\Temp\ipykernel_18240\3516699795.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns


Training the model...

--- Model Evaluation ---
Accuracy: 0.8013

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.53      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409

Confusion Matrix:
[[929 106]
 [174 200]]

Top 5 Most Important Features:
tenure                            0.083255
MonthlyCharges                    0.056304
Contract_Two year                 0.041804
PaymentMethod_Electronic check    0.026520
InternetService_Fiber optic       0.024244
dtype: float64
